In [1]:
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor

# 1. Load Data
df = pd.read_csv("DI_Leather_preprocessed.csv")

# 2. Apply Feature Engineering
df['theoretical_max_production'] = (df['working_hours'] * 60) / df['total_duration_per_shoe_mins']
df['machine_worker_ratio'] = df['machine_count'] / df['no_of_workers']
df['avg_hours_per_worker'] = df['working_hours'] / df['no_of_workers']

# 3. Define Features and Target
target = 'actual_qty'
drop_cols = [
    'planned_qty', 'production_gap_qty', 'bottleneck_flag', 
    'machine_capacity_per_day', 'week_start_date', 'week_end_date', 
    'financial_year'
]
X = df.drop(columns=[target] + drop_cols)
y = df[target]

# 4. Define Preprocessor
categorical_ohe = ['financial_month', 'factory_unit', 'product_family', 'grade', 'order_category', 'size_id']
categorical_target = ['item_name']
numerical_cols = [col for col in X.columns if col not in categorical_ohe + categorical_target]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat_ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_ohe),
        ('cat_te', TargetEncoder(target_type='continuous'), categorical_target) 
    ])

# 5. Build and Train the Pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor), 
    ('model', GradientBoostingRegressor(random_state=42))
])

print("Training final model...")
pipeline.fit(X, y)

# 6. Save the Pipeline (Preprocessor + Model)
model_filename = 'production_model.joblib'
joblib.dump(pipeline, model_filename)
print(f"Model successfully saved to {model_filename}")

Training final model...
Model successfully saved to production_model.joblib
